In [4]:
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import os
import requests
PASSWORD = os.getenv('SNOWSQL_PWD')
print(PASSWORD)

5eWxWv4EvyCkhkY


In [6]:
try:
    ctx = snowflake.connector.connect(
        user='ABHINAVSHARMA2002',
        password=PASSWORD,
        account='qraojwa-yg67137'
    )
    cs = ctx.cursor()
    try:
        cs.execute("CREATE WAREHOUSE IF NOT EXISTS task_1_warehouse_mg")
        cs.execute("CREATE DATABASE IF NOT EXISTS testdb_mg")
        cs.execute("USE DATABASE testdb_mg")
        cs.execute("CREATE SCHEMA IF NOT EXISTS task_2_mg")
        cs.execute("USE WAREHOUSE task_1_warehouse_mg")
        cs.execute("USE SCHEMA task_2_mg")
        ##cs.execute("SHOW VIEWS IN task_2_mg")
        ##print(cs.fetchall())
        runQueries(cs)
        cs.execute('SELECT * FROM "v_order_f" LIMIT 10')
        print(cs.fetchall())
    except Exception as e:
        print(f"Error: {e}") 
    finally:
        cs.close()
        ctx.close()
except Exception as e:
    print(f"Error: {e}")

[(233913, 1737936000000000000, 9, 'External Hard Drive', 'Electronics', 1, 2110.69, 2110.69, 2110.69, 2025, 1, 1, 5, 72, 'Vanessa Fernandez', 'bethrodriguez@yahoo.com', 'Juliamouth', 'Canada'), (897611, 1737936000000000000, 2, 'Smartphone', 'Electronics', 4, 2948.17, 11792.68, 11792.68, 2025, 1, 1, 5, 81, 'Andrew Morris', 'caitlin01@mullins.org', 'Port Hectortown', 'Australia'), (961159, 1738108800000000000, 17, 'Wireless Router', 'Electronics', 4, 1206.46, 4825.84, 4825.84, 2025, 1, 1, 5, 86, 'Heather Scott', 'daniellemorse@gonzalez.com', 'Port Marcus', 'Canada'), (404639, 1738108800000000000, 8, 'Mechanical Keyboard', 'Electronics', 2, 459.38, 918.76, 918.76, 2025, 1, 1, 5, 52, 'Christopher Adams', 'julie23@gmail.com', 'Lake Luke', 'Germany'), (889129, 1738368000000000000, 9, 'External Hard Drive', 'Electronics', 1, 2110.69, 2110.69, 2110.69, 2025, 1, 2, 5, 7, 'Paul Kane', 'paynechristine@cruz-hall.biz', 'Davisberg', 'UK'), (757874, 1738454400000000000, 4, 'Smartwatch', 'Electronics'

In [2]:
def runQueries(cs):    
    query = """
    CREATE OR REPLACE VIEW "v_order_f" AS
    SELECT 
        o."order_id",
        o."date",
        p."product_id",
        p."product_name",
        p."category",
        o."quantity",
        p."price",
        o."quantity" * p."price" AS "total_amount_local",

        -- Apply exchange rate only if price_currency is NOT 'USD'
        CASE 
            WHEN p."price_currency" = 'USD' THEN o."quantity" * p."price"
            ELSE o."quantity" * p."price" * ex."exchange_rate"
        END AS "total_amount_usd",
        
        -- Fiscal Attributes
        YEAR(TO_TIMESTAMP(o."date" / 1000000000)) AS "order_year",
        QUARTER(TO_TIMESTAMP(o."date" / 1000000000)) AS "order_quarter",
        MONTH(TO_TIMESTAMP(o."date" / 1000000000)) AS "order_month",
        WEEK(TO_TIMESTAMP(o."date" / 1000000000)) AS "order_week",

        -- Customer Information
        cust."customer_id",
        cust."customer_name",
        cust."email",
        cust."customer_city",
        cust."country"

    FROM orders o
    JOIN products p ON o."product_id" = p."product_id"
    JOIN customers cust ON o."customer_id" = cust."customer_id"
    
    -- Join exchangerates based on price_currency (only if not USD)
    LEFT JOIN exchangerates ex 
        ON p."price_currency" = ex."source_currency" 
        AND ex."target_currency" = 'USD';
    """

    cs.execute(query)